# Notebook 12 (V4_3) — HKT Increasing-Price / Large-Bubble Calibration

This notebook applies the combined single-country calibration from
`Single_country_production/12_single_country_production_hkt_larger_bubble_share.ipynb`
to the two-country production model.

The single-country parameters are mapped as follows:

| Single-country HKT object | Two-country production object | Value |
|---|---|---:|
| `gamma` | `γ` | 0.25 |
| `pi` | `π_persist` | 0.80 |
| `xi_u` | `ξ_u` | 2.00 |
| `lambda_u` | `ν_u` | 1.50 |
| `lambda_b = xi_b` | seed `ν_b`, `ξ_W` | 0.50 |

The two-country run keeps `common_world_growth=true`, so the model reports both
the literal seed `ν_b=0.50` and the effective selected `ν_b` used after imposing
`G_b=G_W`. The notebook uses only residual-certified horizons; it does not show
provisional long-horizon paths.

In [ ]:
using Pkg

function find_project_dir(start_dir=pwd())
    dir = start_dir
    while true
        isfile(joinpath(dir, "Project.toml")) && return dir
        parent = dirname(dir)
        parent == dir && return start_dir
        dir = parent
    end
end

const PROJECT_DIR = find_project_dir()
Pkg.activate(PROJECT_DIR)

include(joinpath(PROJECT_DIR, "Two_country_production", "TwoCountryProductionOLG.jl"))

using Plots, LaTeXStrings, Printf, Statistics
using Plots.PlotMeasures

gr()

const OUTDIR = joinpath(PROJECT_DIR, "Two_country_production", "outputs_v43",
                        "hkt_increasing_q_large_bubble_share_common_growth")
isdir(OUTDIR) || mkpath(OUTDIR)

 default(size=(900, 560), framestyle=:box, grid=:y, legend=:best,
         fontfamily="Computer Modern", linewidth=2,
         titlefontsize=11, guidefontsize=10, tickfontsize=9, legendfontsize=8,
         left_margin=8mm, right_margin=6mm, top_margin=7mm, bottom_margin=8mm)

println("Project directory: ", PROJECT_DIR)
println("Output directory:  ", OUTDIR)

## 1. Calibration and Residual-Safe Horizon Selection

The main run starts from the certified `T_max = 45`, `n_buffer = 5` calibration
recorded in `outputs_v43/hkt_increasing_q_large_bubble_share_common_growth/calibration_summary.csv`.
If residual certification fails, the notebook falls back to safer shorter horizons.
The reported path is accepted only if all core residual and regularity checks pass.

In [ ]:
const RESID_TOL = 1e-5
const HORIZON_CANDIDATES = [(T_max=45, n_buffer=5),
                            (T_max=40, n_buffer=8),
                            (T_max=35, n_buffer=15),
                            (T_max=30, n_buffer=20),
                            (T_max=25, n_buffer=25),
                            (T_max=20, n_buffer=30),
                            (T_max=10, n_buffer=20),
                            (T_max=5,  n_buffer=10)]

function combo_params(; T_max::Int, n_buffer::Int)
    return ProductionParams(T_max=T_max,
        γ=0.25,
        π_persist=0.80,
        ξ_u=2.0,
        ν_u=1.5,
        ν_b=0.5,
        ξ_W=0.5,
        common_world_growth=true,
        branch_iters=100,
        n_buffer=n_buffer,
        do_global_polish=false)
end

function baseline_params(; T_max::Int, n_buffer::Int)
    return ProductionParams(T_max=T_max,
        branch_iters=100,
        n_buffer=n_buffer)
end

function path_vector(result, field::Symbol)
    return [getfield(s, field) for s in result.u_path]
end

function positive_for_log(x; floor=1e-20)
    return max.(Float64.(x), floor)
end

function market_errors(result::ProductionSimulationResult)
    T = length(result.u_path)
    stock_us = zeros(T); stock_w = zeros(T); bond = zeros(T)
    aggregate_cap = zeros(T); yid_us = zeros(T); yid_w = zeros(T)
    for (i, s) in enumerate(result.u_path)
        stock_us[i] = abs(s.Q_US - s.ω * s.S - s.ω_star * s.S_star)
        stock_w[i] = abs(s.Q_W - (1 - s.ω) * s.S - (1 - s.ω_star) * s.S_star)
        bond[i] = abs(s.θ * s.A + s.θ_US_star * s.A_star)
        aggregate_cap[i] = abs(s.Q_US + s.Q_W -
                               (result.params.β * s.e_US +
                                (result.params.β + result.params.χ) / (1 + result.params.χ) * s.e_W))
        yid_us[i] = abs(s.Y_US - (s.e_US + s.N_US * s.d_US - s.I_US))
        yid_w[i] = abs(s.Y_W - (s.e_W + s.N_W * s.d_W - s.I_W))
    end
    return (; stock_us, stock_w, bond, aggregate_cap, yid_us, yid_w)
end

function decomposition_errors(result, fv, nf)
    Q_US = path_vector(result, :Q_US)
    return (;
        fv_add=max(abs.((fv.V_agg .+ fv.B_agg) .- Q_US)...),
        nfa_add=max(abs.(nf.NFA .- (nf.NFA_fund .+ nf.NFA_bubble))...),
        delta_add=max(abs.(nf.ΔNFA .- (nf.ΔA .+ nf.ΔV .+ nf.ΔB))...)
    )
end

function certified(result, fv, nf)
    errs = decomposition_errors(result, fv, nf)
    result.branch_converged &&
    isfinite(result.max_u_residual) && result.max_u_residual <= RESID_TOL &&
    isfinite(result.max_bgp_residual) && result.max_bgp_residual <= RESID_TOL &&
    result.diagnostics.psi_ok &&
    all(isfinite, fv.v) && all(isfinite, fv.B_per) && all(isfinite, fv.bubble_share) &&
    all(fv.v .> 0) &&
    errs.fv_add <= 1e-7 * max(1.0, maximum(abs.(path_vector(result, :Q_US)))) &&
    errs.nfa_add <= 1e-7 * max(1.0, maximum(abs.(nf.NFA))) &&
    errs.delta_add <= 1e-7 * max(1.0, maximum(abs.(nf.ΔNFA)))
end

function solve_certified_combo()
    rows = NamedTuple[]
    for cand in HORIZON_CANDIDATES
        @printf("Trying combo calibration: T_max=%d, n_buffer=%d\n", cand.T_max, cand.n_buffer)
        t0 = time()
        result = run_production_simulation(combo_params(T_max=cand.T_max, n_buffer=cand.n_buffer);
                                           verbose=false)
        fv = fundamental_value_path(result)
        nf = nfa_decomposition(result)
        elapsed = time() - t0
        ok = certified(result, fv, nf)
        row = (T_max=cand.T_max,
               n_buffer=cand.n_buffer,
               elapsed_sec=elapsed,
               valid_core=ok,
               branch_converged=result.branch_converged,
               max_u_residual=result.max_u_residual,
               max_bgp_residual=result.max_bgp_residual,
               psi_min=result.diagnostics.psi_min,
               equity_weight_min=result.diagnostics.equity_weight_min,
               ν_b_effective=result.params.ν_b,
               q_growth=path_vector(result, :q_US)[end] / path_vector(result, :q_US)[1],
               bubble_share_final=fv.bubble_share[end],
               bubble_share_max=maximum(fv.bubble_share))
        push!(rows, row)
        @printf("  valid=%s  max_u=%.2e  max_bgp=%.2e  q_T/q_1=%.3f  B/Q_T=%.4f  elapsed=%.1fs\n",
                ok, row.max_u_residual, row.max_bgp_residual, row.q_growth,
                row.bubble_share_final, elapsed)
        ok && return (; result, fv, nf, rows, selected=row)
    end
    error("No horizon candidate satisfied the residual-safe certification.")
end

solve_out = solve_certified_combo()
result = solve_out.result
fv = solve_out.fv
nf = solve_out.nf
cert_rows = solve_out.rows
selected = solve_out.selected
p = result.params
T = length(result.u_path)
tt = 1:T

@printf("\nSelected certified horizon: T_max=%d, n_buffer=%d\n", selected.T_max, selected.n_buffer)
@printf("Seed ν_b = %.3f, effective common-growth ν_b = %.6f\n", 0.5, p.ν_b)
@printf("u-branch converged=%s, max ‖F_u‖=%.2e, max BGP residual=%.2e\n",
        result.branch_converged, result.max_u_residual, result.max_bgp_residual)

## 2. Same-Horizon Baseline Comparison

The comparison below uses the default two-country calibration at the same
certified horizon and buffer. This isolates what the combined HKT calibration is
doing to the U.S. all-`u` price path and bubble share.

In [ ]:
baseline = run_production_simulation(baseline_params(T_max=selected.T_max,
                                                            n_buffer=selected.n_buffer);
                                     verbose=false)
baseline_fv = fundamental_value_path(baseline)

combo_q = path_vector(result, :q_US)
combo_d = path_vector(result, :d_US)
combo_Q = path_vector(result, :Q_US)
combo_e = path_vector(result, :e_US)
base_q = path_vector(baseline, :q_US)
base_d = path_vector(baseline, :d_US)

@printf("%-12s %10s %12s %12s %12s %12s\n",
        "case", "T", "q_T/q_1", "B/Q_T", "max B/Q", "max ‖F_u‖")
@printf("%-12s %10d %12.4f %12.4e %12.4e %12.2e\n",
        "baseline", length(baseline.u_path), base_q[end] / base_q[1],
        baseline_fv.bubble_share[end], maximum(baseline_fv.bubble_share),
        baseline.max_u_residual)
@printf("%-12s %10d %12.4f %12.4e %12.4e %12.2e\n",
        "combo", T, combo_q[end] / combo_q[1],
        fv.bubble_share[end], maximum(fv.bubble_share), result.max_u_residual)

if !(combo_q[end] > combo_q[1])
    error("The combo calibration failed the increasing per-variety price check: q_T <= q_1.")
end
if !(fv.bubble_share[end] > baseline_fv.bubble_share[end])
    error("The combo calibration did not raise the endpoint bubble share above baseline.")
end

## 3. Price, Fundamental, and Bubble Paths

In [ ]:
pd_combo = combo_q ./ combo_d
pd_base = base_q ./ base_d

fig_price_1 = plot(tt, combo_q, yscale=:log10, lw=2.5, color=:steelblue,
                   label=L"q_{US}^u\ \mathrm{combo}",
                   xlabel="period t", ylabel="per-variety price (log)",
                   title="U.S. per-variety stock price")
plot!(fig_price_1, tt, base_q, lw=2, ls=:dash, color=:gray35,
      label="baseline, same T")

fig_price_2 = plot(tt, combo_q, yscale=:log10, lw=2.5, color=:steelblue,
                   label=L"q_{US}^u", xlabel="period t",
                   ylabel="per-variety value (log)",
                   title=L"q_{US}^u = v_{US}^u + B_{US}^u")
plot!(fig_price_2, tt, positive_for_log(fv.v), lw=2, ls=:dash, color=:black,
      label=L"v_{US}^u")
plot!(fig_price_2, tt, positive_for_log(fv.B_per), lw=2, ls=:dot, color=:crimson,
      label=L"B_{US}^u")

fig_price_3 = plot(tt, fv.bubble_share, lw=2.5, color=:crimson,
                   label="combo", xlabel="period t", ylabel=L"B/Q",
                   title="Bubble share of U.S. market cap")
plot!(fig_price_3, tt, baseline_fv.bubble_share, lw=2, ls=:dash, color=:gray35,
      label="baseline")

fig_price_4 = plot(tt, pd_combo, yscale=:log10, lw=2.5, color=:purple,
                   label="combo", xlabel="period t", ylabel=L"q_{US}^u/d_{US}^u\ (log)",
                   title="U.S. price-dividend ratio")
plot!(fig_price_4, tt, pd_base, lw=2, ls=:dash, color=:gray35, label="baseline")

fig_price_bubble = plot(fig_price_1, fig_price_2, fig_price_3, fig_price_4,
                        layout=(2, 2), size=(1180, 820), margin=6mm,
                        plot_title="Common-growth two-country calibration: increasing price and larger bubble share")
savefig(fig_price_bubble, joinpath(OUTDIR, "combo_price_bubble_paths.png"))
fig_price_bubble

## 4. Real-Side and NFA Diagnostics

In [ ]:
phi_US = path_vector(result, :φ_US)
phi_W = path_vector(result, :φ_W)
N_US = path_vector(result, :N_US)
N_W = path_vector(result, :N_W)
Y_US = path_vector(result, :Y_US)
Y_W = path_vector(result, :Y_W)
Q_US = path_vector(result, :Q_US)

nfa_gdp = nf.NFA ./ combo_e
bubble_drag_gdp = nf.NFA_bubble ./ combo_e

fig_nfa_1 = plot(tt, phi_US, lw=2.4, color=:steelblue,
                 label=L"\varphi_{US}", xlabel="period t", ylabel="share",
                 title="Production labor allocation")
plot!(fig_nfa_1, tt, phi_W, lw=2, ls=:dash, color=:tomato, label=L"\varphi_W")

fig_nfa_2 = plot(tt, N_US, yscale=:log10, lw=2.4, color=:steelblue,
                 label=L"N_{US}", xlabel="period t", ylabel="knowledge stock (log)",
                 title="Knowledge stocks")
plot!(fig_nfa_2, tt, N_W, lw=2, ls=:dash, color=:tomato, label=L"N_W")

fig_nfa_3 = plot(tt, Q_US, yscale=:log10, lw=2.4, color=:forestgreen,
                 label=L"\mathcal Q_{US}", xlabel="period t", ylabel="aggregate value (log)",
                 title="Aggregate U.S. stock market")
plot!(fig_nfa_3, tt, positive_for_log(fv.V_agg), lw=2, ls=:dash, color=:black,
      label=L"V_Q")
plot!(fig_nfa_3, tt, positive_for_log(fv.B_agg), lw=2, ls=:dot, color=:crimson,
      label=L"B_Q")

fig_nfa_4 = plot(tt, nfa_gdp, lw=2.4, color=:black,
                 label=L"NFA/e_{US}", xlabel="period t", ylabel="ratio",
                 title="U.S. NFA and bubble drag")
plot!(fig_nfa_4, tt, bubble_drag_gdp, lw=2, ls=:dot, color=:crimson,
      label=L"-B_Q/e_{US}")
hline!(fig_nfa_4, [0.0], lc=:gray, ls=:dash, label="")

fig_nfa = plot(fig_nfa_1, fig_nfa_2, fig_nfa_3, fig_nfa_4,
               layout=(2, 2), size=(1180, 820), margin=6mm,
               plot_title="All-u real-side and NFA paths")
savefig(fig_nfa, joinpath(OUTDIR, "combo_nfa_decomposition.png"))
fig_nfa

## 5. Residual, Regularity, and Additivity Checks

In [ ]:
merr = market_errors(result)
derrs = decomposition_errors(result, fv, nf)
u_res = [s.residual_norm for s in result.u_path]
bgp_res = [b.residual_norm for b in result.bgp_seq]
psi_path = [s.Psi for s in result.u_path]

@printf("Residual certification:\n")
@printf("  max u-residual          %.2e\n", maximum(u_res))
@printf("  max BGP residual        %.2e\n", maximum(bgp_res))
@printf("  min Psi                 %.4e\n", minimum(psi_path))
@printf("  equity-weight min slack %.4e\n", result.diagnostics.equity_weight_min)
@printf("Market-clearing max errors:\n")
@printf("  US stock %.2e | RoW stock %.2e | bond %.2e | aggregate cap %.2e\n",
        maximum(merr.stock_us), maximum(merr.stock_w), maximum(merr.bond), maximum(merr.aggregate_cap))
@printf("  Y=e+D-I US %.2e | RoW %.2e\n", maximum(merr.yid_us), maximum(merr.yid_w))
@printf("Additivity errors: V+B-Q %.2e | NFA split %.2e | Δ split %.2e\n",
        derrs.fv_add, derrs.nfa_add, derrs.delta_add)

fig_res_1 = plot(tt, positive_for_log(u_res; floor=1e-16), yscale=:log10,
                 lw=2.3, color=:gray25, label=L"\|F_u\|",
                 xlabel="period t", ylabel="residual (log)",
                 title="u-branch residuals")
hline!(fig_res_1, [RESID_TOL], lc=:red, ls=:dash, label="1e-5")

fig_res_2 = plot(0:(length(bgp_res)-1), positive_for_log(bgp_res; floor=1e-16),
                 yscale=:log10, lw=2.3, color=:steelblue, label="BGP residual",
                 xlabel="BGP index", ylabel="residual (log)", title="Switch-branch BGP residuals")
hline!(fig_res_2, [RESID_TOL], lc=:red, ls=:dash, label="1e-5")

fig_res_3 = plot(tt, psi_path, lw=2.3, color=:forestgreen, label=L"\Psi_t",
                 xlabel="period t", ylabel=L"\Psi_t", title="Effective-kernel regularity")
hline!(fig_res_3, [0.0], lc=:red, ls=:dash, label="")

fig_res_4 = plot(tt, positive_for_log(merr.stock_us; floor=1e-18), yscale=:log10,
                 lw=2.1, label="US stock", xlabel="period t",
                 ylabel="error (log)", title="Market-clearing identities")
plot!(fig_res_4, tt, positive_for_log(merr.stock_w; floor=1e-18), lw=2.1, ls=:dash,
      label="RoW stock")
plot!(fig_res_4, tt, positive_for_log(merr.bond; floor=1e-18), lw=2.1, ls=:dot,
      label="bond")
plot!(fig_res_4, tt, positive_for_log(merr.aggregate_cap; floor=1e-18), lw=2.1, ls=:dashdot,
      label="aggregate cap")

fig_residuals = plot(fig_res_1, fig_res_2, fig_res_3, fig_res_4,
                     layout=(2, 2), size=(1180, 820), margin=6mm,
                     plot_title="Residual-safe certification checks")
savefig(fig_residuals, joinpath(OUTDIR, "combo_residual_checks.png"))
fig_residuals

## 6. 11_v9 all-`u` endogenous path replication

These blocks mirror the 11_v9 all-path diagnostics for the current 12_v9 calibration.

In [ ]:

late_range(T) = max(1, T - 19):T

function pretty(x)
    x === missing && return ""
    x isa Bool && return string(x)
    x isa Integer && return string(x)
    if x isa AbstractFloat
        if !isfinite(x)
            return string(x)
        elseif abs(x) >= 1000 || (abs(x) > 0 && abs(x) < 1e-3)
            return @sprintf("%.3e", x)
        else
            return @sprintf("%.4f", x)
        end
    end
    return string(x)
end

function html_escape(x)
    s = string(x)
    s = replace(s, "&" => "&amp;")
    s = replace(s, "<" => "&lt;")
    s = replace(s, ">" => "&gt;")
    s = replace(s, "\"" => "&quot;")
    return s
end

function show_html_table(rows, cols; maxrows=length(rows))
    shown = rows[1:min(maxrows, length(rows))]
    buf = IOBuffer()
    println(buf, "<div style=\"max-width:100%; overflow-x:auto; margin:0.4rem 0 1rem 0;\">")
    println(buf, "<table style=\"border-collapse:collapse; font-size:12px; line-height:1.25; white-space:nowrap;\">")
    println(buf, "<thead><tr>")
    for c in cols
        println(buf, "<th style=\"border-bottom:1px solid #999; padding:4px 8px; text-align:right;\">", html_escape(c), "</th>")
    end
    println(buf, "</tr></thead><tbody>")
    for row in shown
        println(buf, "<tr>")
        for c in cols
            println(buf, "<td style=\"border-bottom:1px solid #ddd; padding:3px 8px; text-align:right;\">", html_escape(pretty(get(row, c, missing))), "</td>")
        end
        println(buf, "</tr>")
    end
    if length(rows) > length(shown)
        println(buf, "<tr><td style=\"padding:3px 8px;\">...</td>")
        for _ in 2:length(cols)
            println(buf, "<td></td>")
        end
        println(buf, "</tr>")
    end
    println(buf, "</tbody></table></div>")
    display("text/html", String(take!(buf)))
end

function write_columns_csv(path, cols::Vector{String}, columns::Vector{<:AbstractVector})
    @assert !isempty(columns)
    n = length(columns[1])
    @assert all(length(c) == n for c in columns)
    open(path, "w") do io
        println(io, join(cols, ","))
        for i in 1:n
            println(io, join((columns[j][i] for j in eachindex(columns)), ","))
        end
    end
    return path
end

function write_u_path_csv(path, result)
    fields = collect(fieldnames(typeof(result.u_path[1])))
    open(path, "w") do io
        println(io, join(string.(fields), ","))
        for s in result.u_path
            println(io, join((getfield(s, f) for f in fields), ","))
        end
    end
    return path
end

function field_path(result, f::Symbol)
    return path_vector(result, f)
end

function plot_field_group(result, fields::Vector{Symbol}, title_label; cols::Int=2, yscale=:identity)
    rows = cld(length(fields), cols)
    panels = Plots.Plot[]
    for f in fields
        vals = field_path(result, f)
        p_field = plot(tt, vals, lw=2.0, lc=:steelblue, label="",
                       xlabel="period t", ylabel=string(f), title=string(f),
                       yscale=yscale, titlefontsize=10,
                       left_margin=10mm, bottom_margin=8mm)
        hline!(p_field, [0.0], lc=:gray, ls=:dot, label="")
        push!(panels, p_field)
    end
    return plot(panels..., layout=(rows, cols), size=(1160, max(320, 260 * rows)),
                plot_title=title_label, margin=6mm)
end

production_price_fields = [:N_US, :N_W, :φ_US, :φ_W, :Y_US, :Y_W, :e_US, :e_W,
                           :q_US, :q_W, :d_US, :d_W, :Q_US, :Q_W, :I_US, :I_W]
asset_portfolio_fields = [:A, :A_star, :S, :S_star, :ω, :ω_star, :θ, :θ_US_star,
                          :R_f, :R_f_W, :Psi]
returns_u_fields = [:R_US_u, :R_W_u, :R_p_u, :R_A_u, :R_p_star_u, :R_A_star_u]
returns_b_fields = [:R_US_b, :R_W_b, :R_p_b, :R_A_b, :R_p_star_b, :R_A_star_b, :residual_norm]

all_path_csv = write_u_path_csv(joinpath(OUTDIR, "combo_all_u_endogenous_paths.csv"), result)
e_US = field_path(result, :e_US)
Y_US = field_path(result, :Y_US)
println("saved all-u endogenous path CSV to ", all_path_csv)

fig_all_u_1 = plot_field_group(result, production_price_fields, "All-u production, prices, and market values"; cols=2)
savefig(fig_all_u_1, joinpath(OUTDIR, "combo_all_u_production_prices.png"))
fig_all_u_1


In [ ]:

fig_all_u_2 = plot_field_group(result, asset_portfolio_fields, "All-u assets, portfolios, bond prices, and kernel wedge"; cols=2)
savefig(fig_all_u_2, joinpath(OUTDIR, "combo_all_u_assets_portfolios.png"))
fig_all_u_2


In [ ]:

fig_all_u_3 = plot_field_group(result, returns_u_fields, "All-u continuation-branch returns"; cols=2)
savefig(fig_all_u_3, joinpath(OUTDIR, "combo_all_u_returns_continuation.png"))
fig_all_u_3


In [ ]:

fig_all_u_4 = plot_field_group(result, returns_b_fields, "All-u switch-branch returns and residual norm"; cols=2)
savefig(fig_all_u_4, joinpath(OUTDIR, "combo_all_u_returns_switch_residual.png"))

all_fields = collect(fieldnames(typeof(result.u_path[1])))
plotted_fields = vcat(production_price_fields, asset_portfolio_fields, returns_u_fields, returns_b_fields)
unplotted_fields = setdiff(setdiff(all_fields, [:t]), plotted_fields)
println("UPeriodState fields excluding t: ", length(setdiff(all_fields, [:t])))
println("Plotted fields: ", length(plotted_fields))
println("Unplotted fields: ", unplotted_fields)
println("Saved replicated all-u path figures to ", OUTDIR)
fig_all_u_4


## 7. 11_v9 `Q_US/e_US` demand vs production decomposition

This applies the 11_v9 demand-side and production-side `Q/e` split to the 12_v9 calibration.

In [ ]:

function q_demand_production_decomposition(result)
    p = result.params
    T_local = length(result.u_path)
    e_US_local = [s.e_US for s in result.u_path]
    Q_e = [s.Q_US / s.e_US for s in result.u_path]
    Q_us_demand_e = [s.ω * s.S / s.e_US for s in result.u_path]
    Q_row_demand_e = [s.ω_star * s.S_star / s.e_US for s in result.u_path]
    wH_US = [s.q_US * p.a_US * s.N_US for s in result.u_path]
    Q_wage_cap_e = [(wH_US[t] / p.a_US) / e_US_local[t] for t in 1:T_local]
    Q_innovation_cap_e = [((1 - result.u_path[t].φ_US) * p.H_US * wH_US[t]) / e_US_local[t] for t in 1:T_local]
    return (Q_e=Q_e, Q_us_demand_e=Q_us_demand_e, Q_row_demand_e=Q_row_demand_e,
            Q_wage_cap_e=Q_wage_cap_e, Q_innovation_cap_e=Q_innovation_cap_e,
            wH_e=wH_US ./ e_US_local,
            demand_err=maximum(abs.(Q_e .- Q_us_demand_e .- Q_row_demand_e)),
            production_err=maximum(abs.(Q_e .- Q_wage_cap_e .- Q_innovation_cap_e)))
end

qd = q_demand_production_decomposition(result)
q_rows = [Dict{String, Any}(
    "Q_e_final" => qd.Q_e[end],
    "Q_us_demand_e_final" => qd.Q_us_demand_e[end],
    "Q_row_demand_e_final" => qd.Q_row_demand_e[end],
    "Q_wage_cap_e_final" => qd.Q_wage_cap_e[end],
    "Q_innovation_cap_e_final" => qd.Q_innovation_cap_e[end],
    "demand_add_err_max" => qd.demand_err,
    "production_add_err_max" => qd.production_err,
)]
show_html_table(q_rows, ["Q_e_final", "Q_us_demand_e_final", "Q_row_demand_e_final", "Q_wage_cap_e_final", "Q_innovation_cap_e_final", "demand_add_err_max", "production_add_err_max"])

write_columns_csv(joinpath(OUTDIR, "combo_q_demand_production_decomposition.csv"),
    ["t", "Q_e", "Q_us_demand_e", "Q_row_demand_e", "Q_wage_cap_e", "Q_innovation_cap_e", "phi_US", "wH_e"],
    [tt, qd.Q_e, qd.Q_us_demand_e, qd.Q_row_demand_e, qd.Q_wage_cap_e, qd.Q_innovation_cap_e, field_path(result, :φ_US), qd.wH_e])

p_q_total = plot(tt, qd.Q_e, lw=2.6, lc=:steelblue, label=L"\mathcal{Q}_{US}/e_{US}",
                 xlabel="period t", ylabel="ratio", title="US Q/e path")
p_q_demand = plot(tt, qd.Q_e, lw=2.2, lc=:black, ls=:dash, label=L"\mathcal{Q}_{US}/e_{US}",
                  xlabel="period t", ylabel="ratio", title="Demand side")
plot!(p_q_demand, tt, qd.Q_us_demand_e, lw=2.3, lc=:steelblue, label=L"\omega S/e_{US}")
plot!(p_q_demand, tt, qd.Q_row_demand_e, lw=2.3, lc=:tomato, label=L"\omega^*S^*/e_{US}")
p_q_prod = plot(tt, qd.Q_e, lw=2.2, lc=:black, ls=:dash, label=L"\mathcal{Q}_{US}/e_{US}",
                xlabel="period t", ylabel="ratio", title="Production side")
plot!(p_q_prod, tt, qd.Q_wage_cap_e, lw=2.3, lc=:darkgreen, label=L"w_H/(a_{US}e_{US})")
plot!(p_q_prod, tt, qd.Q_innovation_cap_e, lw=2.3, lc=:darkorange, label=L"(1-\varphi_{US})H_{US}w_H/e_{US}")
fig_q_decomp = plot(p_q_total, p_q_demand, p_q_prod, layout=(1,3), size=(1480, 430), margin=8mm)
savefig(fig_q_decomp, joinpath(OUTDIR, "combo_q_demand_production_decomposition.png"))
fig_q_decomp


## 8. 11_v9 NFA component replication

This reproduces the portfolio-position, market-cap, bubble, and NFA component accounting from 11_v9.

In [ ]:

function nfa_components_08(result)
    p = result.params
    nf_local = nfa_decomposition(result)
    fv_local = fundamental_value_path(result)
    home_equity = [(1 - s.ω) * (1 - s.θ) * s.A for s in result.u_path]
    foreign_liability = [-s.ω_star * (1 - s.θ_US_star) * s.A_star for s in result.u_path]
    bond_position = [s.θ * s.A for s in result.u_path]
    nfa_sum = home_equity .+ foreign_liability .+ bond_position
    wH_US = [s.q_US * p.a_US * s.N_US * p.H_US for s in result.u_path]
    e_US_local = [s.e_US for s in result.u_path]
    add_err_path = abs.(nfa_sum .- nf_local.NFA)
    return (nf=nf_local, fv=fv_local, home_equity=home_equity, foreign_liability=foreign_liability,
            bond_position=bond_position, nfa_sum=nfa_sum,
            A_US=[s.A for s in result.u_path], Q_US=nf_local.Q_US, B_agg=fv_local.B_agg,
            V_agg=fv_local.V_agg, wH_US=wH_US,
            omega=[s.ω for s in result.u_path], omega_star=[s.ω_star for s in result.u_path],
            theta=[s.θ for s in result.u_path], theta_US_star=[s.θ_US_star for s in result.u_path],
            add_err=maximum(add_err_path),
            add_err_e=maximum(add_err_path ./ e_US_local))
end

nc = nfa_components_08(result)
show_html_table([Dict{String, Any}(
    "component_add_err_max" => nc.add_err,
    "component_add_err_over_e_max" => nc.add_err_e,
    "NFA_e_final" => nc.nf.NFA[end] / e_US[end],
    "home_equity_e_final" => nc.home_equity[end] / e_US[end],
    "foreign_liability_e_final" => nc.foreign_liability[end] / e_US[end],
    "bond_position_e_final" => nc.bond_position[end] / e_US[end],
    "bubble_drag_e_final" => nc.nf.NFA_bubble[end] / e_US[end],
)], ["component_add_err_max", "component_add_err_over_e_max", "NFA_e_final", "home_equity_e_final", "foreign_liability_e_final", "bond_position_e_final", "bubble_drag_e_final"])

write_columns_csv(joinpath(OUTDIR, "combo_nfa_components_08.csv"),
    ["t", "home_equity", "foreign_liability", "bond_position", "nfa_sum", "NFA", "A_US", "Q_US", "V_agg", "B_agg", "NFA_fund", "NFA_bubble", "NFA_b_cf", "DeltaNFA", "DeltaA", "DeltaV", "DeltaB", "e_US"],
    [tt, nc.home_equity, nc.foreign_liability, nc.bond_position, nc.nfa_sum, nc.nf.NFA, nc.A_US, nc.Q_US, nc.V_agg, nc.B_agg, nc.nf.NFA_fund, nc.nf.NFA_bubble, nc.nf.NFA_b_cf, nc.nf.ΔNFA, nc.nf.ΔA, nc.nf.ΔV, nc.nf.ΔB, e_US])

p_comp = plot(tt, nc.home_equity ./ e_US .* 100, lw=2.3, lc=:steelblue,
              label="home equity / e_US", xlabel="period t", ylabel="% of US GDP",
              title="NFA portfolio-position components", legend=:outerright)
plot!(p_comp, tt, nc.foreign_liability ./ e_US .* 100, lw=2.3, lc=:tomato,
      label="foreign equity liability / e_US")
plot!(p_comp, tt, nc.bond_position ./ e_US .* 100, lw=2.3, lc=:darkgreen, label="bond position / e_US")
plot!(p_comp, tt, nc.nfa_sum ./ e_US .* 100, lw=2.1, lc=:black, ls=:dash, label="sum / e_US")
hline!(p_comp, [0.0], lc=:gray, ls=:dot, label="")

p_assets = plot(tt, nc.A_US ./ e_US .* 100, lw=2.3, lc=:steelblue, label="A / e_US",
                xlabel="period t", ylabel="% of US GDP", title="A, Q, fundamental value, and bubble", legend=:outerright)
plot!(p_assets, tt, nc.Q_US ./ e_US .* 100, lw=2.3, lc=:black, ls=:dash, label="Q_US / e_US")
plot!(p_assets, tt, nc.V_agg ./ e_US .* 100, lw=2.3, lc=:seagreen, label="V_Q / e_US")
plot!(p_assets, tt, nc.B_agg ./ e_US .* 100, lw=2.3, lc=:purple, ls=:dot, label="B_Q / e_US")
plot!(p_assets, tt, nc.wH_US ./ e_US .* 100, lw=2.3, lc=:darkorange, ls=:dashdot, label="w_H H / e_US")
hline!(p_assets, [0.0], lc=:gray, ls=:dot, label="")

p_shares = plot(tt, nc.omega, lw=2.1, lc=:steelblue, label="omega",
                xlabel="period t", ylabel="share", title="Portfolio shares", legend=:outerright)
plot!(p_shares, tt, 1 .- nc.omega, lw=2.1, lc=:navy, ls=:dash, label="1 - omega")
plot!(p_shares, tt, nc.theta, lw=2.1, lc=:darkgreen, label="theta")
plot!(p_shares, tt, 1 .- nc.theta, lw=2.1, lc=:seagreen, ls=:dash, label="1 - theta")
plot!(p_shares, tt, nc.omega_star, lw=2.1, lc=:tomato, label="omega_star")
plot!(p_shares, tt, nc.theta_US_star, lw=2.1, lc=:purple, label="theta_US_star")
plot!(p_shares, tt, 1 .- nc.theta_US_star, lw=2.1, lc=:darkorange, ls=:dash, label="1 - theta_US_star")
hline!(p_shares, [0.0, 1.0], lc=:gray, ls=:dot, label="")

p_bubble_nfa = plot(tt, nc.nf.NFA ./ e_US .* 100, lw=2.4, lc=:black, label="NFA / e_US",
                    xlabel="period t", ylabel="% of US GDP", title="NFA = (A - V_Q) + (-B_Q)", legend=:outerright)
plot!(p_bubble_nfa, tt, nc.nf.NFA_fund ./ e_US .* 100, lw=2.2, lc=:steelblue, ls=:dash, label="(A - V_Q) / e_US")
plot!(p_bubble_nfa, tt, nc.nf.NFA_bubble ./ e_US .* 100, lw=2.2, lc=:red, ls=:dot, label="-B_Q / e_US")
plot!(p_bubble_nfa, tt, nc.nf.NFA_b_cf ./ e_US .* 100, lw=2.0, lc=:gray, ls=:dashdot, label="NFA^b / e_US")
hline!(p_bubble_nfa, [0.0], lc=:gray, ls=:dot, label="")

fig_nfa_components = plot(p_comp, p_assets, p_shares, p_bubble_nfa, layout=(4,1), size=(1160, 1360), margin=8mm)
savefig(fig_nfa_components, joinpath(OUTDIR, "combo_nfa_components_08.png"))
fig_nfa_components


## 9. 11_v9 AHP-style NFA accounting

The AHP-style identity is `NFA_t = q_{W,t} n_{W,t} + b_t - q_{US,t} n^*_{US,t}`. Changes are split into current-account quantity changes and valuation price changes.

In [ ]:

function ahp_style_decomposition(result)
    T_local = length(result.u_path)
    q_US = [s.q_US for s in result.u_path]
    q_W = [s.q_W for s in result.u_path]
    n_W = [(1 - s.ω) * (1 - s.θ) * s.A / s.q_W for s in result.u_path]
    n_US_star = [s.ω_star * (1 - s.θ_US_star) * s.A_star / s.q_US for s in result.u_path]
    bond = [s.θ * s.A for s in result.u_path]
    e_US_local = [s.e_US for s in result.u_path]
    NFA = q_W .* n_W .+ bond .- q_US .* n_US_star
    CA = zeros(T_local)
    VA = zeros(T_local)
    RES = zeros(T_local)
    for t in 2:T_local
        VA[t] = n_W[t - 1] * (q_W[t] - q_W[t - 1]) -
                n_US_star[t - 1] * (q_US[t] - q_US[t - 1])
        CA[t] = q_W[t] * (n_W[t] - n_W[t - 1]) -
                q_US[t] * (n_US_star[t] - n_US_star[t - 1]) +
                (bond[t] - bond[t - 1])
        RES[t] = NFA[t] - NFA[t - 1] - CA[t] - VA[t]
    end
    cum_CA = cumsum(CA)
    cum_VA = cumsum(VA)
    cum_RES = cumsum(RES)
    accounting_line = NFA[1] .+ cum_CA .+ cum_VA .+ cum_RES
    nfa_path = nfa_decomposition(result).NFA
    nfa_err_path = abs.(NFA .- nfa_path)
    return (q_US=q_US, q_W=q_W, n_W=n_W, n_US_star=n_US_star, bond=bond,
            NFA=NFA, CA=CA, VA=VA, RES=RES,
            cum_CA=cum_CA, cum_VA=cum_VA, cum_RES=cum_RES,
            accounting_line=accounting_line,
            residual_max=maximum(abs.(RES[2:end])),
            nfa_identity_err=maximum(nfa_err_path),
            nfa_identity_err_e=maximum(nfa_err_path ./ e_US_local))
end

ahp = ahp_style_decomposition(result)
show_html_table([Dict{String, Any}(
    "NFA_identity_err_max" => ahp.nfa_identity_err,
    "NFA_identity_err_over_e_max" => ahp.nfa_identity_err_e,
    "AHP_RES_abs_max" => ahp.residual_max,
    "NFA_Y_final" => ahp.NFA[end] / Y_US[end],
    "cum_CA_Y_final" => ahp.cum_CA[end] / Y_US[end],
    "cum_VA_Y_final" => ahp.cum_VA[end] / Y_US[end],
    "cum_RES_Y_final" => ahp.cum_RES[end] / Y_US[end],
)], ["NFA_identity_err_max", "NFA_identity_err_over_e_max", "AHP_RES_abs_max", "NFA_Y_final", "cum_CA_Y_final", "cum_VA_Y_final", "cum_RES_Y_final"])

write_columns_csv(joinpath(OUTDIR, "combo_ahp_style_decomposition.csv"),
    ["t", "q_W", "q_US", "n_W", "n_US_star", "bond", "NFA", "CA", "VA", "RES", "cum_CA", "cum_VA", "cum_RES", "accounting_line", "Y_US", "e_US"],
    [tt, ahp.q_W, ahp.q_US, ahp.n_W, ahp.n_US_star, ahp.bond, ahp.NFA, ahp.CA, ahp.VA, ahp.RES, ahp.cum_CA, ahp.cum_VA, ahp.cum_RES, ahp.accounting_line, Y_US, e_US])

p_ahp_fig2 = plot(tt, ahp.NFA ./ Y_US, lw=2.6, lc=:black, label=L"NFA_t/Y_{US,t}",
                  xlabel="period t", ylabel="fraction of US Y", title="AHP-style cumulative accounting", legend=:outerright)
plot!(p_ahp_fig2, tt, ahp.cum_CA ./ Y_US, lw=2.2, lc=:steelblue, label=L"\sum CA/Y_{US,t}")
plot!(p_ahp_fig2, tt, ahp.cum_VA ./ Y_US, lw=2.2, lc=:tomato, label=L"\sum VA/Y_{US,t}")
plot!(p_ahp_fig2, tt, ahp.cum_RES ./ Y_US, lw=2.0, lc=:purple, ls=:dot, label=L"\sum RES/Y_{US,t}")
plot!(p_ahp_fig2, tt, ahp.accounting_line ./ Y_US, lw=1.8, lc=:darkgreen, ls=:dash, label=L"(NFA_1+\sum CA+\sum VA+\sum RES)/Y_{US,t}")
hline!(p_ahp_fig2, [0.0], lc=:gray, ls=:dot, label="")

p_ahp_change = plot(tt, (ahp.NFA .- ahp.NFA[1]) ./ Y_US, lw=2.6, lc=:black,
                    label=L"(NFA_t-NFA_1)/Y_{US,t}", xlabel="period t", ylabel="fraction of US Y",
                    title="Change from first path period", legend=:outerright)
plot!(p_ahp_change, tt, ahp.cum_CA ./ Y_US, lw=2.2, lc=:steelblue, label=L"\sum CA/Y_{US,t}")
plot!(p_ahp_change, tt, ahp.cum_VA ./ Y_US, lw=2.2, lc=:tomato, label=L"\sum VA/Y_{US,t}")
plot!(p_ahp_change, tt, (ahp.cum_CA .+ ahp.cum_VA .+ ahp.cum_RES) ./ Y_US,
      lw=1.8, lc=:darkgreen, ls=:dash, label=L"\sum(CA+VA+RES)/Y_{US,t}")
hline!(p_ahp_change, [0.0], lc=:gray, ls=:dot, label="")

p_ahp_flows = plot(tt, ahp.CA ./ Y_US .* 100, lw=2.2, lc=:steelblue, label=L"CA_t/Y_{US,t}",
                   xlabel="period t", ylabel="% of US Y", title="Period flows", legend=:outerright)
plot!(p_ahp_flows, tt, ahp.VA ./ Y_US .* 100, lw=2.2, lc=:tomato, label=L"VA_t/Y_{US,t}")
plot!(p_ahp_flows, tt, ahp.RES ./ Y_US .* 100, lw=1.8, lc=:purple, ls=:dot, label=L"RES_t/Y_{US,t}")
hline!(p_ahp_flows, [0.0], lc=:gray, ls=:dot, label="")

p_ahp_positions = plot(tt, (ahp.q_W .* ahp.n_W) ./ Y_US .* 100, lw=2.3, lc=:steelblue,
                       label=L"q_W n_W/Y_{US}", xlabel="period t", ylabel="% of US Y",
                       title="AHP gross-position pieces", legend=:outerright)
plot!(p_ahp_positions, tt, -(ahp.q_US .* ahp.n_US_star) ./ Y_US .* 100,
      lw=2.3, lc=:tomato, label=L"-q_{US}n^*_{US}/Y_{US}")
plot!(p_ahp_positions, tt, ahp.bond ./ Y_US .* 100, lw=2.3, lc=:darkgreen, label=L"b/Y_{US}")
plot!(p_ahp_positions, tt, ahp.NFA ./ Y_US .* 100, lw=2.1, lc=:black, ls=:dash, label=L"NFA/Y_{US}")
hline!(p_ahp_positions, [0.0], lc=:gray, ls=:dot, label="")

fig_ahp = plot(p_ahp_fig2, p_ahp_change, p_ahp_flows, p_ahp_positions,
               layout=(4,1), size=(1160, 1360), margin=8mm)
savefig(fig_ahp, joinpath(OUTDIR, "combo_ahp_style_decomposition.png"))
fig_ahp


### AHP two-panel component summary

In [ ]:

p_ahp_components = plot(tt, ahp.cum_VA ./ Y_US, lw=2.4, lc=:tomato,
                        label="cum VA / Y_US", xlabel="period t",
                        ylabel="fraction of US Y", title="AHP components", legend=:outerright)
plot!(p_ahp_components, tt, ahp.cum_CA ./ Y_US, lw=2.4, lc=:steelblue,
      label="cum CA / Y_US")
plot!(p_ahp_components, tt, ahp.cum_RES ./ Y_US, lw=2.2, lc=:purple, ls=:dot,
      label="cum RES / Y_US")
hline!(p_ahp_components, [0.0], lc=:gray, ls=:dot, label="")

p_ahp_reconstruction = plot(tt, ahp.NFA ./ Y_US, lw=2.6, lc=:black,
                            label="NFA / Y_US", xlabel="period t",
                            ylabel="fraction of US Y", title="NFA and component sum", legend=:outerright)
plot!(p_ahp_reconstruction, tt, ahp.accounting_line ./ Y_US, lw=2.2,
      lc=:darkgreen, ls=:dash, label="NFA_1 + cum components / Y_US")
hline!(p_ahp_reconstruction, [0.0], lc=:gray, ls=:dot, label="")

fig_ahp_two_panel = plot(p_ahp_components, p_ahp_reconstruction,
                         layout=(1,2), size=(1280, 430), margin=8mm)
savefig(fig_ahp_two_panel, joinpath(OUTDIR, "combo_ahp_style_two_panel_components.png"))
fig_ahp_two_panel


### AHP VA and CA subcomponents scaled by `Y_US`

In [ ]:

function ahp_va_ca_subcomponents(result, ahp)
    T_local = length(result.u_path)
    q_US = [s.q_US for s in result.u_path]
    q_W = [s.q_W for s in result.u_path]
    n_W = [(1 - s.ω) * (1 - s.θ) * s.A / s.q_W for s in result.u_path]
    n_US_star = [s.ω_star * (1 - s.θ_US_star) * s.A_star / s.q_US for s in result.u_path]
    bond = [s.θ * s.A for s in result.u_path]

    VA_row_equity = zeros(T_local)
    VA_us_liability = zeros(T_local)
    CA_row_equity = zeros(T_local)
    CA_us_liability = zeros(T_local)
    CA_bond = zeros(T_local)

    for t in 2:T_local
        VA_row_equity[t] = n_W[t - 1] * (q_W[t] - q_W[t - 1])
        VA_us_liability[t] = -n_US_star[t - 1] * (q_US[t] - q_US[t - 1])
        CA_row_equity[t] = q_W[t] * (n_W[t] - n_W[t - 1])
        CA_us_liability[t] = -q_US[t] * (n_US_star[t] - n_US_star[t - 1])
        CA_bond[t] = bond[t] - bond[t - 1]
    end

    return (VA_row_equity=VA_row_equity,
            VA_us_liability=VA_us_liability,
            CA_row_equity=CA_row_equity,
            CA_us_liability=CA_us_liability,
            CA_bond=CA_bond,
            VA_add_err=maximum(abs.(VA_row_equity .+ VA_us_liability .- ahp.VA)),
            CA_add_err=maximum(abs.(CA_row_equity .+ CA_us_liability .+ CA_bond .- ahp.CA)))
end

ahp_sub = ahp_va_ca_subcomponents(result, ahp)
show_html_table([Dict{String, Any}(
    "VA_subcomponent_add_err_max" => ahp_sub.VA_add_err,
    "CA_subcomponent_add_err_max" => ahp_sub.CA_add_err,
)], ["VA_subcomponent_add_err_max", "CA_subcomponent_add_err_max"])

write_columns_csv(joinpath(OUTDIR, "combo_ahp_va_ca_subcomponents_y.csv"),
    ["t", "VA_row_equity_Y", "VA_us_liability_Y", "VA_total_Y", "CA_row_equity_Y", "CA_us_liability_Y", "CA_bond_Y", "CA_total_Y", "Y_US"],
    [tt,
     ahp_sub.VA_row_equity ./ Y_US,
     ahp_sub.VA_us_liability ./ Y_US,
     ahp.VA ./ Y_US,
     ahp_sub.CA_row_equity ./ Y_US,
     ahp_sub.CA_us_liability ./ Y_US,
     ahp_sub.CA_bond ./ Y_US,
     ahp.CA ./ Y_US,
     Y_US])

p_va_sub_y = plot(tt, ahp_sub.VA_row_equity ./ Y_US, lw=2.3, lc=:steelblue,
                  label="US-held RoW equity valuation / Y_US",
                  xlabel="period t", ylabel="fraction of US Y",
                  title="VA subcomponents / Y_US", legend=:outerright)
plot!(p_va_sub_y, tt, ahp_sub.VA_us_liability ./ Y_US, lw=2.3, lc=:tomato,
      label="foreign-held US equity valuation / Y_US")
plot!(p_va_sub_y, tt, ahp.VA ./ Y_US, lw=2.0, lc=:black, ls=:dash,
      label="VA total / Y_US")
hline!(p_va_sub_y, [0.0], lc=:gray, ls=:dot, label="")

p_ca_sub_y = plot(tt, ahp_sub.CA_row_equity ./ Y_US, lw=2.3, lc=:steelblue,
                  label="US RoW-equity purchases / Y_US",
                  xlabel="period t", ylabel="fraction of US Y",
                  title="CA subcomponents / Y_US", legend=:outerright)
plot!(p_ca_sub_y, tt, ahp_sub.CA_us_liability ./ Y_US, lw=2.3, lc=:tomato,
      label="foreign US-equity purchases / Y_US")
plot!(p_ca_sub_y, tt, ahp_sub.CA_bond ./ Y_US, lw=2.3, lc=:darkgreen,
      label="bond-position change / Y_US")
plot!(p_ca_sub_y, tt, ahp.CA ./ Y_US, lw=2.0, lc=:black, ls=:dash,
      label="CA total / Y_US")
hline!(p_ca_sub_y, [0.0], lc=:gray, ls=:dot, label="")

fig_ahp_va_ca_subcomponents_y = plot(p_va_sub_y, p_ca_sub_y,
                                     layout=(1,2), size=(1380, 450), margin=8mm)
savefig(fig_ahp_va_ca_subcomponents_y, joinpath(OUTDIR, "combo_ahp_va_ca_subcomponents_y.png"))
fig_ahp_va_ca_subcomponents_y


### Cumulative AHP VA and CA subcomponents scaled by `Y_US`

In [ ]:

cum_VA_row_equity = cumsum(ahp_sub.VA_row_equity)
cum_VA_us_liability = cumsum(ahp_sub.VA_us_liability)
cum_CA_row_equity = cumsum(ahp_sub.CA_row_equity)
cum_CA_us_liability = cumsum(ahp_sub.CA_us_liability)
cum_CA_bond = cumsum(ahp_sub.CA_bond)

write_columns_csv(joinpath(OUTDIR, "combo_ahp_cumulative_va_ca_subcomponents_y.csv"),
    ["t", "cum_VA_row_equity_Y", "cum_VA_us_liability_Y", "cum_VA_total_Y", "cum_CA_row_equity_Y", "cum_CA_us_liability_Y", "cum_CA_bond_Y", "cum_CA_total_Y", "Y_US"],
    [tt,
     cum_VA_row_equity ./ Y_US,
     cum_VA_us_liability ./ Y_US,
     ahp.cum_VA ./ Y_US,
     cum_CA_row_equity ./ Y_US,
     cum_CA_us_liability ./ Y_US,
     cum_CA_bond ./ Y_US,
     ahp.cum_CA ./ Y_US,
     Y_US])

p_cum_va_sub_y = plot(tt, cum_VA_row_equity ./ Y_US, lw=2.3, lc=:steelblue,
                      label="cum US-held RoW equity valuation / Y_US",
                      xlabel="period t", ylabel="fraction of US Y",
                      title="Cumulative VA subcomponents / Y_US", legend=:outerright)
plot!(p_cum_va_sub_y, tt, cum_VA_us_liability ./ Y_US, lw=2.3, lc=:tomato,
      label="cum foreign-held US equity valuation / Y_US")
plot!(p_cum_va_sub_y, tt, ahp.cum_VA ./ Y_US, lw=2.0, lc=:black, ls=:dash,
      label="cum VA total / Y_US")
hline!(p_cum_va_sub_y, [0.0], lc=:gray, ls=:dot, label="")

p_cum_ca_sub_y = plot(tt, cum_CA_row_equity ./ Y_US, lw=2.3, lc=:steelblue,
                      label="cum US RoW-equity purchases / Y_US",
                      xlabel="period t", ylabel="fraction of US Y",
                      title="Cumulative CA subcomponents / Y_US", legend=:outerright)
plot!(p_cum_ca_sub_y, tt, cum_CA_us_liability ./ Y_US, lw=2.3, lc=:tomato,
      label="cum foreign US-equity purchases / Y_US")
plot!(p_cum_ca_sub_y, tt, cum_CA_bond ./ Y_US, lw=2.3, lc=:darkgreen,
      label="cum bond-position change / Y_US")
plot!(p_cum_ca_sub_y, tt, ahp.cum_CA ./ Y_US, lw=2.0, lc=:black, ls=:dash,
      label="cum CA total / Y_US")
hline!(p_cum_ca_sub_y, [0.0], lc=:gray, ls=:dot, label="")

fig_ahp_cumulative_va_ca_subcomponents_y = plot(p_cum_va_sub_y, p_cum_ca_sub_y,
                                                layout=(1,2), size=(1450, 450), margin=8mm)
savefig(fig_ahp_cumulative_va_ca_subcomponents_y, joinpath(OUTDIR, "combo_ahp_cumulative_va_ca_subcomponents_y.png"))
fig_ahp_cumulative_va_ca_subcomponents_y


### AHP two-panel component summary scaled by `e_US`

In [ ]:

p_ahp_components_e = plot(tt, ahp.cum_VA ./ e_US, lw=2.4, lc=:tomato,
                          label="cum VA / e_US", xlabel="period t",
                          ylabel="fraction of US e", title="AHP components / e_US", legend=:outerright)
plot!(p_ahp_components_e, tt, ahp.cum_CA ./ e_US, lw=2.4, lc=:steelblue,
      label="cum CA / e_US")
plot!(p_ahp_components_e, tt, ahp.cum_RES ./ e_US, lw=2.2, lc=:purple, ls=:dot,
      label="cum RES / e_US")
hline!(p_ahp_components_e, [0.0], lc=:gray, ls=:dot, label="")

p_ahp_reconstruction_e = plot(tt, ahp.NFA ./ e_US, lw=2.6, lc=:black,
                              label="NFA / e_US", xlabel="period t",
                              ylabel="fraction of US e", title="NFA and component sum / e_US", legend=:outerright)
plot!(p_ahp_reconstruction_e, tt, ahp.accounting_line ./ e_US, lw=2.2,
      lc=:darkgreen, ls=:dash, label="NFA_1 + cum components / e_US")
hline!(p_ahp_reconstruction_e, [0.0], lc=:gray, ls=:dot, label="")

fig_ahp_two_panel_e = plot(p_ahp_components_e, p_ahp_reconstruction_e,
                           layout=(1,2), size=(1280, 430), margin=8mm)
savefig(fig_ahp_two_panel_e, joinpath(OUTDIR, "combo_ahp_style_two_panel_components_e.png"))
fig_ahp_two_panel_e


## 10. Export Tables and Residual Report

In [ ]:
function write_calibration_summary(filename)
    open(filename, "w") do io
        println(io, "name,value")
        println(io, "gamma_seed,0.25")
        println(io, "pi_persist_seed,0.80")
        println(io, "xi_u_seed,2.0")
        println(io, "nu_u_seed,1.5")
        println(io, "nu_b_seed,0.5")
        println(io, "xi_W_seed,0.5")
        println(io, "common_world_growth,true")
        println(io, "nu_b_effective,$(p.ν_b)")
        println(io, "T_max,$T")
        println(io, "n_buffer,$(selected.n_buffer)")
        println(io, "branch_iters,$(p.branch_iters)")
        println(io, "valid_core,$(selected.valid_core)")
        println(io, "max_u_residual,$(result.max_u_residual)")
        println(io, "max_bgp_residual,$(result.max_bgp_residual)")
        println(io, "baseline_q_growth,$(base_q[end] / base_q[1])")
        println(io, "combo_q_growth,$(combo_q[end] / combo_q[1])")
        println(io, "baseline_bubble_share_final,$(baseline_fv.bubble_share[end])")
        println(io, "combo_bubble_share_final,$(fv.bubble_share[end])")
    end
    return filename
end

function write_all_u_paths(filename)
    open(filename, "w") do io
        println(io, "t,N_US,N_W,phi_US,phi_W,q_US,q_W,d_US,d_W,Q_US,Q_W,e_US,e_W,Y_US,Y_W,omega,omega_star,theta_US_star,theta,R_f,R_f_W,Psi,residual_norm,v_US,B_per,V_agg,B_agg,bubble_share,NFA,NFA_fund,NFA_bubble,NFA_b_cf,dNFA,dA,dV,dB")
        for t in 1:T
            s = result.u_path[t]
            vals = [t, s.N_US, s.N_W, s.φ_US, s.φ_W, s.q_US, s.q_W, s.d_US, s.d_W,
                    s.Q_US, s.Q_W, s.e_US, s.e_W, s.Y_US, s.Y_W, s.ω, s.ω_star,
                    s.θ_US_star, s.θ, s.R_f, s.R_f_W, s.Psi, s.residual_norm,
                    fv.v[t], fv.B_per[t], fv.V_agg[t], fv.B_agg[t], fv.bubble_share[t],
                    nf.NFA[t], nf.NFA_fund[t], nf.NFA_bubble[t], nf.NFA_b_cf[t],
                    nf.ΔNFA[t], nf.ΔA[t], nf.ΔV[t], nf.ΔB[t]]
            println(io, join(string.(vals), ","))
        end
    end
    return filename
end

function write_horizon_attempts(filename)
    open(filename, "w") do io
        println(io, "T_max,n_buffer,elapsed_sec,valid_core,branch_converged,max_u_residual,max_bgp_residual,psi_min,equity_weight_min,nu_b_effective,q_growth,bubble_share_final,bubble_share_max")
        for row in cert_rows
            println(io, join(string.([row.T_max, row.n_buffer, row.elapsed_sec, row.valid_core,
                                      row.branch_converged, row.max_u_residual, row.max_bgp_residual,
                                      row.psi_min, row.equity_weight_min, row.ν_b_effective,
                                      row.q_growth, row.bubble_share_final, row.bubble_share_max]), ","))
        end
    end
    return filename
end

function write_residual_report(filename)
    open(filename, "w") do io
        write(io, "HKT INCREASING-Q / LARGE-BUBBLE TWO-COUNTRY PRODUCTION REPORT\n")
        write(io, "================================================================\n\n")
        @printf(io, "Selected T_max = %d, n_buffer = %d, branch_iters = %d\n", T, selected.n_buffer, p.branch_iters)
        @printf(io, "common_world_growth = %s; seed ν_b = %.6f; effective ν_b = %.6f\n", p.common_world_growth, 0.5, p.ν_b)
        @printf(io, "γ = %.6f, π = %.6f, ξ_u = %.6f, ν_u = %.6f, ξ_W = %.6f\n\n",
                p.γ, p.π_persist, p.ξ_u, p.ν_u, p.ξ_W)

        write(io, "Horizon attempts:\n")
        for row in cert_rows
            @printf(io, "  T=%2d buffer=%2d valid=%s max_u=%.2e max_bgp=%.2e q_T/q_1=%.4f B/Q_T=%.4e elapsed=%.1fs\n",
                    row.T_max, row.n_buffer, row.valid_core, row.max_u_residual,
                    row.max_bgp_residual, row.q_growth, row.bubble_share_final,
                    row.elapsed_sec)
        end

        write(io, "\nBaseline comparison at selected horizon:\n")
        @printf(io, "  baseline q_T/q_1 = %.6f, B/Q_T = %.6e, max B/Q = %.6e, max_u = %.2e\n",
                base_q[end] / base_q[1], baseline_fv.bubble_share[end],
                maximum(baseline_fv.bubble_share), baseline.max_u_residual)
        @printf(io, "  combo    q_T/q_1 = %.6f, B/Q_T = %.6e, max B/Q = %.6e, max_u = %.2e\n",
                combo_q[end] / combo_q[1], fv.bubble_share[end], maximum(fv.bubble_share),
                result.max_u_residual)

        write(io, "\nResidual and regularity checks:\n")
        @printf(io, "  u-branch converged = %s\n", result.branch_converged)
        @printf(io, "  max u residual = %.2e\n", maximum(u_res))
        @printf(io, "  max BGP residual = %.2e\n", maximum(bgp_res))
        @printf(io, "  min Ψ = %.6e; psi_ok = %s\n", minimum(psi_path), result.diagnostics.psi_ok)
        @printf(io, "  equity-weight min slack = %.6e; ok = %s\n",
                result.diagnostics.equity_weight_min, result.diagnostics.equity_weights_ok)

        write(io, "\nMarket clearing / identities:\n")
        @printf(io, "  US stock-clearing max = %.2e\n", maximum(merr.stock_us))
        @printf(io, "  RoW stock-clearing max = %.2e\n", maximum(merr.stock_w))
        @printf(io, "  bond-clearing max = %.2e\n", maximum(merr.bond))
        @printf(io, "  aggregate-cap identity max = %.2e\n", maximum(merr.aggregate_cap))
        @printf(io, "  Y=e+D-I US max = %.2e\n", maximum(merr.yid_us))
        @printf(io, "  Y=e+D-I RoW max = %.2e\n", maximum(merr.yid_w))
        @printf(io, "  V+B-Q additivity max = %.2e\n", derrs.fv_add)
        @printf(io, "  NFA split additivity max = %.2e\n", derrs.nfa_add)
        @printf(io, "  Delta split additivity max = %.2e\n", derrs.delta_add)

        write(io, "\nReplicated 11_v9 diagnostics:\n")
        @printf(io, "  Q/e demand additivity max = %.2e\n", qd.demand_err)
        @printf(io, "  Q/e production additivity max = %.2e\n", qd.production_err)
        @printf(io, "  NFA position component additivity max = %.2e (%.2e of e_US)\n", nc.add_err, nc.add_err_e)
        @printf(io, "  AHP NFA identity max = %.2e (%.2e of e_US)\n", ahp.nfa_identity_err, ahp.nfa_identity_err_e)
        @printf(io, "  AHP period residual max = %.2e\n", ahp.residual_max)
        @printf(io, "  AHP VA subcomponent additivity max = %.2e\n", ahp_sub.VA_add_err)
        @printf(io, "  AHP CA subcomponent additivity max = %.2e\n", ahp_sub.CA_add_err)
    end
    return filename
end

calibration_csv = write_calibration_summary(joinpath(OUTDIR, "calibration_summary.csv"))
paths_csv = write_all_u_paths(joinpath(OUTDIR, "combo_all_u_paths.csv"))
attempts_csv = write_horizon_attempts(joinpath(OUTDIR, "horizon_attempts.csv"))
report_txt = write_residual_report(joinpath(OUTDIR, "residual_report.txt"))

println("Saved:")
println("  ", calibration_csv)
println("  ", paths_csv)
println("  ", attempts_csv)
println("  ", report_txt)
println("  ", joinpath(OUTDIR, "combo_price_bubble_paths.png"))
println("  ", joinpath(OUTDIR, "combo_nfa_decomposition.png"))
println("  ", joinpath(OUTDIR, "combo_residual_checks.png"))
println("  ", joinpath(OUTDIR, "combo_all_u_endogenous_paths.csv"))
println("  ", joinpath(OUTDIR, "combo_all_u_production_prices.png"))
println("  ", joinpath(OUTDIR, "combo_all_u_assets_portfolios.png"))
println("  ", joinpath(OUTDIR, "combo_all_u_returns_continuation.png"))
println("  ", joinpath(OUTDIR, "combo_all_u_returns_switch_residual.png"))
println("  ", joinpath(OUTDIR, "combo_q_demand_production_decomposition.csv"))
println("  ", joinpath(OUTDIR, "combo_q_demand_production_decomposition.png"))
println("  ", joinpath(OUTDIR, "combo_nfa_components_08.csv"))
println("  ", joinpath(OUTDIR, "combo_nfa_components_08.png"))
println("  ", joinpath(OUTDIR, "combo_ahp_style_decomposition.csv"))
println("  ", joinpath(OUTDIR, "combo_ahp_style_decomposition.png"))
println("  ", joinpath(OUTDIR, "combo_ahp_style_two_panel_components.png"))
println("  ", joinpath(OUTDIR, "combo_ahp_style_two_panel_components_e.png"))
println("  ", joinpath(OUTDIR, "combo_ahp_va_ca_subcomponents_y.csv"))
println("  ", joinpath(OUTDIR, "combo_ahp_va_ca_subcomponents_y.png"))
println("  ", joinpath(OUTDIR, "combo_ahp_cumulative_va_ca_subcomponents_y.csv"))
println("  ", joinpath(OUTDIR, "combo_ahp_cumulative_va_ca_subcomponents_y.png"))

## Summary

The selected common-growth two-country calibration is residual-certified on the
reported horizon. Relative to the same-horizon baseline, it produces an increasing
U.S. per-variety stock price and a materially larger bubble share while preserving
model residual, regularity, and accounting checks.

The notebook now also replicates the 11_v9 all-`u` endogenous path plots,
`Q_US/e_US` demand/production decomposition, NFA component decomposition, and
AHP-style NFA accounting blocks for this 12_v9 calibration.